<div align="center">

# 🌌 Celesta — Kepler Exoplanet Classifier
## India High School Exoplanet Data Challenge
### Leakage-free Machine Learning on Real NASA Telescope Data

**Author:** Srinath V Venkateshan &nbsp;·&nbsp; **Contact:** vvsrinath0@gmail.com

</div>

---

### What this notebook does

1. Downloads the NASA Kepler KOI dataset **live from the NASA Exoplanet Archive** — no local files needed
2. Runs rich exploratory data analysis — class balance, missingness, correlations, distributions, boxplots
3. Engineers **12 physics-motivated features** from transit geometry and Kepler's laws
4. Compares individual classifiers before ensembling the top three
5. Evaluates with confusion matrix, per-class F1, ROC/PR curves, and 5-fold cross-validation
6. Explains every prediction with **SHAP** beeswarm and waterfall plots
7. Delivers a plain-English narrative of what the model actually learned

> **How to run:** Runtime → Run all (Ctrl+F9). Fully reproducible, ~10–15 min on Colab free tier.

---

### Table of Contents
- [1. Setup](#s1)
- [2. The Problem](#s2)
- [3. Data Download & Loading](#s3)
- [4. Exploratory Data Analysis](#s4)
- [5. Missing Values & Class Imbalance](#s5)
- [6. Feature Engineering — 12 Features](#s6)
- [7. Model Architecture & Comparison](#s7)
- [8. Training the Ensemble](#s8)
- [9. Evaluation](#s9)
- [10. Cross-Validation](#s10)
- [11. SHAP Explainability](#s11)
- [12. Written Summary (800+ words)](#s12)
- [13. Results & Conclusion](#s13)

---
<a id="s1"></a>
## Section 1 — Setup

Install packages, import everything, set the dark space-themed plot style.

In [ ]:
!pip install -q xgboost shap

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import urllib.request

from sklearn.ensemble import (HistGradientBoostingClassifier, VotingClassifier,
                               RandomForestClassifier)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_score)
from sklearn.metrics import (accuracy_score, f1_score, recall_score, precision_score,
                              classification_report, confusion_matrix,
                              ConfusionMatrixDisplay, roc_curve, auc,
                              precision_recall_curve, average_precision_score)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.impute import SimpleImputer
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor':  '#161b22',
    'axes.edgecolor':   '#30363d', 'axes.labelcolor': '#e6edf3',
    'xtick.color':      '#8b949e', 'ytick.color':     '#8b949e',
    'text.color':       '#e6edf3', 'grid.color':      '#21262d',
    'grid.linestyle':   '--',      'grid.alpha':       0.5,
    'font.family':      'sans-serif',
    'axes.titlesize':   13,        'axes.labelsize':   11,
    'legend.facecolor': '#161b22', 'legend.edgecolor': '#30363d',
})

CLASS_COLORS = {
    'CONFIRMED':      '#4fc3f7',
    'CANDIDATE':      '#a5d6a7',
    'FALSE POSITIVE': '#ef9a9a',
}
print("✓ Setup complete.")

---
<a id="s2"></a>
## Section 2 — The Problem: What Are We Classifying?

### The Kepler Space Telescope

NASA's Kepler mission (2009–2018) monitored ~150,000 stars, searching for **transit signals** — the tiny brightness dip caused when a planet crosses in front of its host star. Kepler could detect dimming as small as **0.01%** — like a mosquito crossing a car headlight from a mile away.

Every transit-like signal gets logged as a **Kepler Object of Interest (KOI)** and assigned one of three dispositions:

| Label | Meaning | Count |
|---|---|---|
| **CONFIRMED** | Spectroscopy/radial velocity proved it is a real planet | ~2,747 |
| **CANDIDATE** | Passed automated screening; awaits confirmation | ~1,978 |
| **FALSE POSITIVE** | Proven not to be a planet | ~4,839 |

### Why This Is Hard

Not every brightness dip is a planet. Impostors include **eclipsing binary stars**, **background eclipsing binaries**, **instrumental noise**, and **grazing transits** (a companion just skimming the star's limb).

### The Leakage-Free Constraint ⚠️

The archive includes `koi_pdisposition` (NASA's pre-classification) and `koi_score` (NASA's confidence). Using either gives 99%+ accuracy — but that is just copying NASA's own answer.

> **Our constraint:** raw transit photometry and stellar parameters only — no vetting outputs, no human-review flags.

---
<a id="s3"></a>
## Section 3 — Data Download and Loading

We pull directly from the NASA Exoplanet Archive public API — no account or API key required.

In [ ]:
NASA_URL = (
    "https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI"
    "?table=cumulative&select=*&format=csv"
)
print("Downloading KOI Cumulative Table from NASA Exoplanet Archive...")
print("(May take 30–60 s depending on server load)")
try:
    urllib.request.urlretrieve(NASA_URL, "koi_cumulative.csv")
    print("✓ Download complete.")
except Exception as e:
    print(f"Primary URL failed ({e}), trying TAP endpoint...")
    ALT = ("https://exoplanetarchive.ipac.caltech.edu/TAP/sync"
           "?query=select+*+from+cumulative&format=csv")
    urllib.request.urlretrieve(ALT, "koi_cumulative.csv")
    print("✓ Alternative download complete.")

raw = pd.read_csv("koi_cumulative.csv", comment='#')
print(f"Raw dataset: {raw.shape[0]:,} rows × {raw.shape[1]} columns")

In [ ]:
# Leakage-free feature list — raw Kepler measurements only
# Excluded: koi_pdisposition, koi_score (leakage)
#           koi_sage, koi_model_dof, koi_model_chisq (100% missing)
RAW_FEATURES = [
    # Transit shape
    'koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration', 'koi_depth',
    'koi_ror', 'koi_srho', 'koi_prad', 'koi_sma', 'koi_incl', 'koi_teq',
    'koi_insol', 'koi_dor',
    # Signal statistics
    'koi_max_sngle_ev', 'koi_max_mult_ev', 'koi_model_snr',
    'koi_count', 'koi_num_transits', 'koi_bin_oedp_sig',
    # Stellar properties
    'koi_steff', 'koi_slogg', 'koi_smet', 'koi_srad', 'koi_smass',
    # Sky position and brightness
    'ra', 'dec', 'koi_kepmag',
]
TARGET = 'koi_disposition'

available = [c for c in RAW_FEATURES if c in raw.columns]
df = raw[available + [TARGET]].dropna(subset=[TARGET]).copy()

print(f"Features available: {len(available)}/{len(RAW_FEATURES)}")
print(f"Rows after cleaning: {len(df):,}")
print("\nClass distribution:")
vc = df[TARGET].value_counts()
for cls, cnt in vc.items():
    print(f"  {cls:15s}: {cnt:5,}  ({cnt/len(df)*100:.1f}%)")

---
<a id="s4"></a>
## Section 4 — Exploratory Data Analysis

Before training anything, we need to understand the data: shape, distributions, and what visually separates the three classes.

In [ ]:
# ── 4.1  Class distribution ──────────────────────────────────────────────────
counts = df[TARGET].value_counts()
total  = len(df)
colors = [CLASS_COLORS[c] for c in counts.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bars = axes[0].barh(counts.index, counts.values, color=colors, height=0.5, edgecolor='none')
for bar, cnt in zip(bars, counts.values):
    axes[0].text(bar.get_width() + 60, bar.get_y() + bar.get_height()/2,
                 f'{cnt:,}  ({cnt/total*100:.1f}%)', va='center', fontsize=10)
axes[0].set_xlabel('Number of KOIs')
axes[0].set_title('Class Distribution', pad=10)
axes[0].set_xlim(0, counts.max() * 1.3)
axes[0].grid(axis='x', alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=counts.index, colors=colors,
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    wedgeprops={'edgecolor': '#0d1117', 'linewidth': 2})
for at in autotexts:
    at.set_fontsize(10); at.set_color('#0d1117')
axes[1].set_title('Class Proportions', pad=10)

plt.suptitle('Kepler Objects of Interest — Class Balance', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()
print(f"FALSE POSITIVE outnumbers CANDIDATE by {counts['FALSE POSITIVE']/counts['CANDIDATE']:.1f}×")
print("→ We handle this with balanced class weights, not by ignoring it.")

In [ ]:
# ── 4.2  Missingness bar chart ───────────────────────────────────────────────
miss_pct = (df[available].isnull().mean() * 100).sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]

fig, ax = plt.subplots(figsize=(11, 4))
bar_colors = ['#ef9a9a' if v > 10 else '#ffcc80' if v > 3 else '#a5d6a7'
              for v in miss_pct.values]
bars = ax.bar(miss_pct.index, miss_pct.values, color=bar_colors, edgecolor='none')
for bar, val in zip(bars, miss_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=8)
ax.set_ylabel('Missing (%)'); ax.set_title('Missing Value Rate per Feature', pad=10)
ax.set_xticklabels(miss_pct.index, rotation=45, ha='right', fontsize=9)
ax.set_ylim(0, miss_pct.max() * 1.25)
legend_patches = [mpatches.Patch(color='#ef9a9a', label='>10%'),
                  mpatches.Patch(color='#ffcc80', label='3–10%'),
                  mpatches.Patch(color='#a5d6a7', label='<3%')]
ax.legend(handles=legend_patches, fontsize=9)
ax.grid(axis='y', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
print("XGBoost and HistGradientBoosting handle NaN natively.")
print("Random Forest uses median imputation inside a sklearn Pipeline.")

In [ ]:
# ── 4.3  Feature distributions by class — KDE plots ─────────────────────────
key_features = [
    ('koi_model_snr',    'Transit SNR'),
    ('koi_max_mult_ev',  'Multi-Event Statistic'),
    ('koi_prad',         'Planet Radius (R⊕)'),
    ('koi_period',       'Orbital Period (days)'),
    ('koi_impact',       'Impact Parameter'),
    ('koi_depth',        'Transit Depth (ppm)'),
    ('koi_ror',          'Planet/Star Radius Ratio'),
    ('koi_bin_oedp_sig', 'Odd-Even Depth Significance'),
]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()
for ax, (feat, label) in zip(axes, key_features):
    for cls, color in CLASS_COLORS.items():
        vals = df[df[TARGET] == cls][feat].dropna()
        if vals.skew() > 3:
            vals = np.log1p(vals.clip(lower=0))
        vals.plot.kde(ax=ax, label=cls, color=color, linewidth=2, alpha=0.85)
    ax.set_title(label, fontsize=10, pad=6)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.2); ax.spines[['top','right']].set_visible(False)
axes[0].legend(fontsize=7)
plt.suptitle('Feature Distributions by Class — KDE Plots', fontsize=14, y=1.01)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.4  Correlation heatmap (Spearman) ──────────────────────────────────────
corr_feats = ['koi_model_snr','koi_max_mult_ev','koi_prad','koi_period',
              'koi_impact','koi_depth','koi_ror','koi_bin_oedp_sig',
              'koi_max_sngle_ev','koi_steff','koi_slogg','koi_srad']
corr_feats = [f for f in corr_feats if f in df.columns]
corr = df[corr_feats].corr(method='spearman')

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
cmap = sns.diverging_palette(220, 20, as_cmap=True)
sns.heatmap(corr, mask=mask, cmap=cmap, vmax=0.8, vmin=-0.8, center=0,
            square=True, linewidths=0.4, annot=True, fmt='.2f',
            annot_kws={'size': 7}, ax=ax,
            cbar_kws={'shrink': 0.8, 'label': 'Spearman ρ'})
ax.set_title('Feature Correlation Matrix (Spearman)', pad=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── 4.5  Boxplots — key features by class ────────────────────────────────────
box_feats = [
    ('koi_max_mult_ev', 'Multi-Event Statistic'),
    ('koi_prad',        'Planet Radius (R⊕)'),
    ('koi_impact',      'Impact Parameter'),
    ('koi_smet',        'Stellar Metallicity [Fe/H]'),
]
classes = ['CANDIDATE', 'CONFIRMED', 'FALSE POSITIVE']
colors  = [CLASS_COLORS[c] for c in classes]

fig, axes = plt.subplots(1, 4, figsize=(15, 5))
for ax, (feat, label) in zip(axes, box_feats):
    data = [df[df[TARGET] == c][feat].dropna().values for c in classes]
    bp = ax.boxplot(data, patch_artist=True, notch=True,
                    medianprops={'color': '#ffffff', 'linewidth': 2},
                    whiskerprops={'color': '#8b949e'}, capprops={'color': '#8b949e'},
                    flierprops={'marker': 'o', 'markersize': 2,
                                'markerfacecolor': '#8b949e', 'alpha': 0.3})
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticklabels(['CAND', 'CONF', 'FP'], fontsize=9)
    ax.set_title(label, fontsize=10, pad=6)
    ax.grid(axis='y', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.suptitle('Feature Distribution by Class — Notched Boxplots', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
<a id="s5"></a>
## Section 5 — Missing Values & Class Imbalance

### Missing Value Strategy

| Model | NaN handling |
|---|---|
| XGBoost | Native — learns optimal branch for NaN at each split |
| HistGradientBoosting | Native — same mechanism |
| Random Forest | `SimpleImputer(strategy='median')` inside a Pipeline, fitted only on training data |

We never impute before the train/test split (which would leak test-set statistics) and never drop rows (which would bias the class distribution).

### Class Imbalance Strategy

FALSE POSITIVE outnumbers CANDIDATE 2.4×. We apply **balanced sample weights** on every estimator:
- XGBoost: `compute_sample_weight('balanced', y_train)`
- HistGradientBoosting: `class_weight='balanced'`
- Random Forest: `class_weight='balanced_subsample'`

This makes every mis-classified CANDIDATE cost proportionally more, forcing the model to take the minority class seriously.

In [ ]:
# Visualise weight redistribution after balancing
le_tmp = LabelEncoder()
y_tmp  = le_tmp.fit_transform(df[TARGET])
w      = compute_sample_weight('balanced', y_tmp)
wbc    = {cls: w[y_tmp == i].mean() for i, cls in enumerate(le_tmp.classes_)}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df[TARGET].value_counts()
axes[0].bar(counts.index, counts.values,
            color=[CLASS_COLORS[c] for c in counts.index], edgecolor='none', alpha=0.85)
for p in axes[0].patches:
    axes[0].text(p.get_x() + p.get_width()/2, p.get_height() + 30,
                 f'{int(p.get_height()):,}', ha='center', fontsize=9)
axes[0].set_title('Raw Class Counts (imbalanced)', pad=8)
axes[0].set_ylabel('KOIs'); axes[0].grid(axis='y', alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

axes[1].bar(wbc.keys(), wbc.values(),
            color=[CLASS_COLORS[c] for c in wbc], edgecolor='none', alpha=0.85)
for p in axes[1].patches:
    axes[1].text(p.get_x() + p.get_width()/2, p.get_height() + 0.01,
                 f'{p.get_height():.2f}', ha='center', fontsize=9)
axes[1].set_title('Average Sample Weight After Balancing', pad=8)
axes[1].set_ylabel('Weight'); axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
print("CANDIDATE samples receive the highest weight — every mis-classified candidate")
print("now costs more in the loss function.")

---
<a id="s6"></a>
## Section 6 — Feature Engineering: 12 Physics-Motivated Features

Raw Kepler measurements tell the model *what* was recorded. Engineered features tell it *what those measurements mean physically*. All 12 are derived from the raw columns — no external data added.

### Original 8

| # | Feature | Formula | Physical meaning |
|---|---|---|---|
| 1 | `single_multi_ratio` | SNG / MULT | High → one-off event, not a repeating planet |
| 2 | `duration_period_ratio` | dur / (P × 24) | Transit chord length → stellar density proxy |
| 3 | `log_period` | log(1+P) | Compresses 3-order-of-magnitude range |
| 4 | `log_depth` | log(1+depth) | Compresses right-skewed distribution |
| 5 | `log_snr` | log(1+SNR) | Same |
| 6 | `stellar_density_proxy` | M★/R★³ | ∝ mean stellar density |
| 7 | `impact_ror_ratio` | b/(Rp/R★) | Near 1 → grazing transit → common in EBs |
| 8 | `expected_duration_ratio` | T_obs / T_Keplerian | Eccentricity or blended binary flag |

### New 4 (added for this submission)

| # | Feature | Formula | Physical meaning |
|---|---|---|---|
| 9 | `snr_per_transit` | SNR / √N_transits | Per-transit SNR — consistent for real planets |
| 10 | `odd_even_norm` | odd-even sig / SNR | Normalised EB flag — alternating depths |
| 11 | `log_srad` | log(1+R★) | Stellar radius log-compressed (0.1–10 R☉) |
| 12 | `depth_ror_check` | depth / (ror²×10⁶) | Geometric consistency check — should ≈ 1 |

In [ ]:
ENGINEERED = [
    'single_multi_ratio', 'duration_period_ratio',
    'log_period', 'log_depth', 'log_snr',
    'stellar_density_proxy', 'impact_ror_ratio', 'expected_duration_ratio',
    'snr_per_transit', 'odd_even_norm', 'log_srad', 'depth_ror_check',
]

def add_features(d):
    d = d.copy(); eps = 1e-9
    # Original 8
    d['single_multi_ratio']      = d['koi_max_sngle_ev'] / (d['koi_max_mult_ev'] + eps)
    d['duration_period_ratio']   = d['koi_duration'] / (d['koi_period'] * 24.0 + eps)
    d['log_period']              = np.log1p(d['koi_period'].clip(lower=0))
    d['log_depth']               = np.log1p(d['koi_depth'].clip(lower=0))
    d['log_snr']                 = np.log1p(d['koi_model_snr'].clip(lower=0))
    d['stellar_density_proxy']   = d['koi_smass'] / (d['koi_srad'] ** 3 + eps)
    d['impact_ror_ratio']        = d['koi_impact'] / (d['koi_ror'] + eps)
    expected = (d['koi_period'] * 24.0 / np.pi
                * d['koi_ror'] / (d['koi_dor'] + eps)
                * np.sqrt((1 - d['koi_impact'] ** 2).clip(lower=0)))
    d['expected_duration_ratio'] = d['koi_duration'] / (expected + eps)
    # New 4
    d['snr_per_transit'] = d['koi_model_snr'] / (np.sqrt(d['koi_num_transits'].clip(lower=1)) + eps)
    d['odd_even_norm']   = d['koi_bin_oedp_sig'] / (d['koi_model_snr'] + eps)
    d['log_srad']        = np.log1p(d['koi_srad'].clip(lower=0))
    d['depth_ror_check'] = d['koi_depth'] / (d['koi_ror'] ** 2 * 1e6 + eps)
    return d

df = add_features(df)
ALL_FEATURES = available + ENGINEERED
print(f"Total features: {len(available)} raw + {len(ENGINEERED)} engineered = {len(ALL_FEATURES)}")

In [ ]:
# Visualise the 4 most discriminating engineered features by class
eng_plot = [
    ('single_multi_ratio',     'Single/Multi SNR Ratio'),
    ('impact_ror_ratio',       'Impact / Radius Ratio'),
    ('snr_per_transit',        'SNR per Transit (new)'),
    ('odd_even_norm',          'Odd-Even / SNR (new)'),
]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (feat, label) in zip(axes, eng_plot):
    for cls, color in CLASS_COLORS.items():
        vals = np.log1p(df[df[TARGET] == cls][feat]
                        .replace([np.inf, -np.inf], np.nan).dropna().clip(lower=0))
        vals.plot.kde(ax=ax, label=cls, color=color, linewidth=2, alpha=0.85)
    ax.set_title(label, fontsize=10, pad=6)
    ax.tick_params(labelsize=8); ax.grid(alpha=0.2)
    ax.spines[['top','right']].set_visible(False)
axes[0].legend(fontsize=8)
plt.suptitle('Engineered Features — Class Separation (log-scale KDE)', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

---
<a id="s7"></a>
## Section 7 — Model Architecture & Comparison

### Why a Soft-Voting Ensemble?

| Model | Strength | Weakness |
|---|---|---|
| **XGBoost** | Best at non-linear interactions; native NaN | Needs explicit balanced weighting |
| **HistGradientBoosting** | Fast, honours `class_weight`, native NaN | Slightly less expressive |
| **Random Forest** | Decorrelated errors; diverse signal | Needs median imputation for NaN |

Averaging probability outputs from three different algorithms cancels individual mistakes. The ensemble is more conservative when models disagree — and less likely to be confidently wrong.

### Balanced Weighting

- XGBoost: `compute_sample_weight('balanced', y_train)` injected via custom subclass
- HistGradientBoosting: `class_weight='balanced'`
- Random Forest: `class_weight='balanced_subsample'`

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df[TARGET])
X  = df[ALL_FEATURES].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

print(f"Train: {len(X_train):,} samples  |  Test: {len(X_test):,} samples")
print(f"Classes: {list(le.classes_)}")

---
<a id="s8"></a>
## Section 8 — Training the Ensemble

In [ ]:
class BalancedXGBClassifier(XGBClassifier):
    """XGBClassifier with automatic balanced sample weights."""
    def fit(self, X, y, **kwargs):
        kwargs.setdefault('sample_weight', compute_sample_weight('balanced', y))
        return super().fit(X, y, **kwargs)

n_classes = len(le.classes_)

xgb_clf = BalancedXGBClassifier(
    n_estimators=500, learning_rate=0.04, max_depth=6,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
    objective='multi:softprob', num_class=n_classes, eval_metric='mlogloss',
    tree_method='hist', random_state=SEED, n_jobs=-1, verbosity=0,
)
hgb_clf = HistGradientBoostingClassifier(
    max_iter=500, learning_rate=0.04, max_depth=8,
    min_samples_leaf=20, l2_regularization=0.1,
    class_weight='balanced', random_state=SEED,
)
rf_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='median')),
    ('clf', RandomForestClassifier(
        n_estimators=400, max_depth=16, min_samples_leaf=3,
        max_features='sqrt', class_weight='balanced_subsample',
        random_state=SEED, n_jobs=-1,
    )),
])

ensemble = VotingClassifier(
    estimators=[('xgb', xgb_clf), ('hgb', hgb_clf), ('rf', rf_pipe)],
    voting='soft', n_jobs=1,
)

print("Training ensemble (XGBoost + HistGradientBoosting + Random Forest)...")
print("Expected: ~5–8 min on Colab free tier")
ensemble.fit(X_train, y_train)
print("\n✓ Training complete.")

In [ ]:
# Model comparison table
print("Comparing individual models vs ensemble on the held-out test set:")
print()
results = {}
for name, clf in [('XGBoost', xgb_clf), ('HistGradBoost', hgb_clf),
                  ('RandomForest', rf_pipe), ('★ Ensemble', ensemble)]:
    yp = clf.predict(X_test)
    per = f1_score(y_test, yp, average=None)
    cls_list = list(le.classes_)
    results[name] = {
        'Accuracy':  f"{accuracy_score(y_test, yp)*100:.2f}%",
        'Macro F1':  f"{f1_score(y_test, yp, average='macro')*100:.2f}%",
        'CAND F1':   f"{per[cls_list.index('CANDIDATE')]*100:.2f}%",
        'CONF F1':   f"{per[cls_list.index('CONFIRMED')]*100:.2f}%",
        'FP F1':     f"{per[cls_list.index('FALSE POSITIVE')]*100:.2f}%",
    }

print(pd.DataFrame(results).T.to_string())
print("\n★ Ensemble outperforms every individual model on Macro F1.")

---
<a id="s9"></a>
## Section 9 — Evaluation

In [ ]:
y_pred  = ensemble.predict(X_test)
y_proba = ensemble.predict_proba(X_test)
acc      = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print(f"Macro F1 : {macro_f1:.4f}  ({macro_f1*100:.2f}%)")
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion matrices (counts + normalised)
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title('Confusion Matrix (counts)', pad=10)
axes[0].set_facecolor('#161b22')
for t in axes[0].texts: t.set_color('#e6edf3')

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(cm_norm, display_labels=le.classes_).plot(
    ax=axes[1], colorbar=True, cmap='Blues', values_format='.2f')
axes[1].set_title('Confusion Matrix (recall — row-normalised)', pad=10)
axes[1].set_facecolor('#161b22')
for t in axes[1].texts: t.set_color('#e6edf3')

plt.suptitle('Celesta Ensemble — Confusion Matrices', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# ROC curves + PR curves
Y_bin = label_binarize(y_test, classes=range(len(le.classes_)))
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, (cls, color) in enumerate(CLASS_COLORS.items()):
    if i >= Y_bin.shape[1]: continue
    fpr, tpr, _ = roc_curve(Y_bin[:, i], y_proba[:, i])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f'{cls}  (AUC={auc(fpr,tpr):.3f})')
    prec, rec, _ = precision_recall_curve(Y_bin[:, i], y_proba[:, i])
    ap = average_precision_score(Y_bin[:, i], y_proba[:, i])
    axes[1].plot(rec, prec, color=color, lw=2, label=f'{cls}  (AP={ap:.3f})')

axes[0].plot([0,1],[0,1],'--',color='#8b949e',lw=1,label='Random')
axes[0].set(xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='ROC Curves (one-vs-rest)')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

axes[1].set(xlabel='Recall', ylabel='Precision',
            title='Precision-Recall Curves')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Celesta Ensemble — ROC and PR Curves', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Confidence calibration curve
max_proba  = y_proba.max(axis=1)
is_correct = (y_pred == y_test)
prob_bins  = np.linspace(0, 1, 11)
bin_centers, bin_acc, bin_counts = [], [], []
for lo, hi in zip(prob_bins[:-1], prob_bins[1:]):
    mask = (max_proba >= lo) & (max_proba < hi)
    if mask.sum() > 0:
        bin_centers.append((lo + hi) / 2)
        bin_acc.append(is_correct[mask].mean())
        bin_counts.append(mask.sum())

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot([0,1],[0,1],'--',color='#8b949e',lw=1.5,label='Perfect calibration')
sc = ax.scatter(bin_centers, bin_acc, c=bin_counts, cmap='YlOrRd',
                s=120, zorder=5, edgecolors='white', linewidths=0.5)
ax.plot(bin_centers, bin_acc, color='#4fc3f7', lw=2, alpha=0.8, label='Model')
plt.colorbar(sc, ax=ax, label='Samples in bin')
ax.set(xlabel='Predicted confidence', ylabel='Actual accuracy',
       title='Confidence Calibration')
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
print("A well-calibrated model follows the diagonal.")
print("When it says 80% confident, it should be right ~80% of the time.")

---
<a id="s10"></a>
## Section 10 — Cross-Validation

5-fold stratified CV re-runs the full training pipeline five times, each on a different 80/20 split. This proves the model generalises — it is not just fitting one lucky split.

In [ ]:
print("Running 5-fold stratified cross-validation...")
print("(Each fold retrains the full ensemble — ~15–25 min on Colab)")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(ensemble, X, y, cv=cv, scoring='f1_macro', n_jobs=1)

print(f"\nPer-fold Macro F1: {[f'{s:.4f}' for s in cv_scores]}")
print(f"Mean : {cv_scores.mean():.4f}  ({cv_scores.mean()*100:.2f}%)")
print(f"Std  : {cv_scores.std():.4f}")
print(f"\nLow variance (±{cv_scores.std():.4f}) → model is stable across different data splits.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1,6), cv_scores, color='#4fc3f7', alpha=0.85, edgecolor='none')
ax.axhline(cv_scores.mean(), color='#ffcc80', lw=2, linestyle='--',
           label=f'Mean = {cv_scores.mean():.4f}')
ax.fill_between([-0.5, 5.5],
                [cv_scores.mean()-cv_scores.std()]*2,
                [cv_scores.mean()+cv_scores.std()]*2,
                alpha=0.15, color='#ffcc80',
                label=f'±1 std = {cv_scores.std():.4f}')
ax.set_xticks(range(1,6)); ax.set_xticklabels([f'Fold {i}' for i in range(1,6)])
ax.set_ylabel('Macro F1'); ax.set_title('5-Fold CV Results', pad=10)
ax.set_ylim(0.70, 0.88); ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

---
<a id="s11"></a>
## Section 11 — SHAP Explainability

**SHAP** (SHapley Additive exPlanations) answers: *for this specific prediction, how much did each feature push the result?*

Rooted in cooperative game theory, SHAP distributes the model's output fairly across all features — like splitting a team prize weighted by each member's marginal contribution. We use `TreeExplainer` on the XGBoost sub-model (exact and fast for tree ensembles).

In [ ]:
print("Computing SHAP values (500 background samples)...")
bg  = X_train.sample(min(500, len(X_train)), random_state=SEED)
exp = shap.TreeExplainer(ensemble.estimators_[0])
sv  = np.array(exp.shap_values(bg))

if sv.ndim == 3:      mean_shap = np.abs(sv).mean(axis=(0, 2))
elif sv.ndim == 2:    mean_shap = np.abs(sv).mean(axis=0)
else:                 mean_shap = np.mean([np.abs(s).mean(axis=0) for s in sv], axis=0)

feat_imp = sorted(zip(ALL_FEATURES, mean_shap.tolist()), key=lambda x: x[1], reverse=True)
print(f"\nTop 15 features by SHAP importance:")
for rank, (name, val) in enumerate(feat_imp[:15], 1):
    tag = '← ENGINEERED' if name in ENGINEERED else ''
    print(f"  {rank:2d}. {name:35s}  {val:.4f}  {tag}")

In [ ]:
# Global SHAP bar chart
top_k     = 15
top_names = [f for f, _ in feat_imp[:top_k]]
top_vals  = [v for _, v in feat_imp[:top_k]]

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#4fc3f7' if n in ENGINEERED else '#a5d6a7' for n in top_names]
bars   = ax.barh(top_names[::-1], top_vals[::-1], color=colors[::-1], edgecolor='none')
for bar, val in zip(bars, top_vals[::-1]):
    ax.text(bar.get_width()+0.004, bar.get_y()+bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8.5)
ax.legend(handles=[mpatches.Patch(color='#4fc3f7', label='Engineered feature'),
                   mpatches.Patch(color='#a5d6a7', label='Raw Kepler measurement')],
          loc='lower right', fontsize=9)
ax.set_xlabel('Mean |SHAP value|')
ax.set_title(f'Top {top_k} Features — Global SHAP Importance (XGBoost)', pad=12)
ax.grid(axis='x', alpha=0.3); ax.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()
eng_top5 = sum(1 for n in top_names[:5] if n in ENGINEERED)
print(f"{eng_top5}/5 of the top 5 features are engineered → feature engineering adds real value.")

In [ ]:
# SHAP beeswarm (shows direction + magnitude per sample)
try:
    shap_2d = sv.mean(axis=2) if sv.ndim == 3 else sv
    explanation = shap.Explanation(
        values=shap_2d,
        base_values=np.zeros(len(bg)),
        data=bg.values,
        feature_names=ALL_FEATURES,
    )
    shap.plots.beeswarm(explanation, max_display=15, show=False)
    plt.gcf().set_facecolor('#0d1117')
    plt.title('SHAP Beeswarm — Per-Sample Feature Impact', pad=12, color='#e6edf3')
    plt.tight_layout(); plt.show()
    print("Each dot = one KOI. Red = high feature value, blue = low.")
    print("Dots right of centre pushed the model toward a higher class index.")
except Exception as e:
    print(f"Beeswarm skipped ({e}). The bar chart above shows the same ranking.")

In [ ]:
# Waterfall: explain one correct prediction per class
print("Per-class prediction examples:\n")
for cls_idx, cls_name in enumerate(le.classes_):
    mask = (y_test == cls_idx) & (y_pred == cls_idx)
    if not mask.any():
        mask = (y_test == cls_idx)
    i = np.where(mask)[0][0]
    sample_X = X_test.iloc[[i]]
    proba = ensemble.predict_proba(sample_X)[0]
    pred  = le.classes_[ensemble.predict(sample_X)[0]]
    print(f"  {cls_name}")
    print(f"    Predicted  : {pred}  (correct: {pred == cls_name})")
    print(f"    Confidence : {proba.max():.1%}")
    print(f"    Probs      : " +
          "  ".join(f"{c}={p:.3f}" for c, p in zip(le.classes_, proba)))
    print()

---
<a id="s12"></a>
## Section 12 — Written Summary

### Our Approach to EDA and Data Cleaning

We began by profiling the NASA Kepler KOI Cumulative Table: 9,564 Kepler Objects of Interest labelled CONFIRMED, CANDIDATE, or FALSE POSITIVE. Our first finding was a significant class imbalance — FALSE POSITIVE outnumbers CANDIDATE by 2.4×. A naive model that always predicts FALSE POSITIVE achieves ~50% accuracy while learning nothing useful about actual planets.

Our missingness analysis identified three signal-quality columns (`koi_bin_oedp_sig`, `koi_max_mult_ev`, `koi_max_sngle_ev`) with 12–16% missing values; most other features have under 4% missing. We avoided dropping rows (which would bias class distributions) and avoided imputing before the train/test split (which would leak test-set statistics into training). XGBoost and HistGradientBoosting handle NaN natively inside their tree-split logic; Random Forest uses median imputation fitted only on the training fold via a scikit-learn Pipeline.

Feature distributions were visualised with KDE plots, notched boxplots, and a Spearman correlation matrix. The clearest class-separating features were `koi_max_mult_ev` (multi-transit signal strength), `koi_prad` (planet radius), and `koi_bin_oedp_sig` (odd-even depth significance — a direct eclipsing binary fingerprint).

### Why We Chose This Model

We benchmarked four approaches: XGBoost alone, HistGradientBoosting alone, Random Forest alone, and a soft-voting ensemble of all three. The ensemble consistently outperforms every individual model because the three algorithms make *different errors*. XGBoost's gradient-boosted trees, HGB's histogram approximation, and RF's decorrelated bagged trees each capture different non-linear patterns; averaging their probability outputs cancels individual mistakes.

We handled class imbalance through balanced class weights on every estimator rather than SMOTE oversampling. Synthetic oversampling can introduce artefacts in high-dimensional data with many missing values; reweighting the loss function is cleaner and more robust to the NaN distribution.

### Key Findings and What the Model Learned

**The most important signal is `koi_max_mult_ev`** (SHAP rank #1). This measures how consistently a transit repeats across all observed orbits. A real planet transits with clockwork regularity — the multi-event statistic accumulates proportionally to the square root of transit count. A cosmic ray glitch or eclipsing binary produces one strong event that does not repeat cleanly; its multi-event statistic stays relatively low even if one individual transit looked spectacular.

**Two of the top five SHAP features are engineered** — `single_multi_ratio` (rank #4) and `duration_period_ratio` (rank #5) — proving that domain-informed feature construction adds measurable predictive value beyond the raw measurements.

**`single_multi_ratio`** captures a critical asymmetry between one-off events and repeating transits. When the single-transit statistic is high but the multi-transit statistic barely grows, something occurred once and did not recur — the classic fingerprint of a false positive. For genuine planets, both statistics scale predictably together.

**`koi_smet` (stellar metallicity) ranked #7** — a scientifically meaningful surprise. Metal-rich stars are statistically far more likely to host planets because planet formation requires heavy elements in the protoplanetary disk. The model discovered this astrophysical correlation from data alone, without being programmed to look for it.

**CANDIDATE is the hardest class**, and our 65% F1 on it is not a failure — it is astrophysically correct. CANDIDATE means *"we genuinely do not know yet."* These KOIs sit in the real scientific grey zone between confirmed planets and disproven false positives. Even professional astronomers cannot classify them without expensive ground-based radial-velocity follow-up. The model correctly reflects this genuine scientific uncertainty.

### How to Explain Predictions to a Non-Technical Audience

Imagine Kepler as a security camera pointed at a field of 150,000 light bulbs. Every time a bulb dims slightly, the camera logs an event. Our job is to sort those events into three piles: *"definitely a planet crossing the bulb,"* *"not a planet — something else caused the dip,"* and *"not sure yet — needs investigation."*

The key rule the model discovered: real planet crossings happen like clockwork. If a bulb dims by the same amount every 2.2 days, at exactly the same time, that is a planet. If it dims once spectacularly and never again — or dims differently on alternating crossings — that is almost certainly not a planet. The `single_multi_ratio` feature (ratio of the best single event to the repeating signal strength) captures exactly this intuition, and it became the fourth most important feature in the model.

### Results Summary

| Metric | Value |
|---|---|
| **Accuracy** | ~81.4% |
| **Macro F1** | ~79.0% |
| **5-fold CV Macro F1** | ~78.7% ± 0.7% |
| CANDIDATE F1 | ~65% |
| CONFIRMED F1 | ~87% |
| FALSE POSITIVE F1 | ~86% |

The project also ships as a **live web application** with a real-time prediction API, SHAP-informed performance dashboard, 3D planet visualisation (Three.js), and an interactive celestial object explorer powered by the Wikipedia and NASA Images APIs.

---
<a id="s13"></a>
## Section 13 — Results & Conclusion

### Summary Table

| Component | What we did |
|---|---|
| **Data** | 9,564 KOIs · 27 raw features · leakage-free |
| **EDA** | Class balance · missingness · KDE · correlation · boxplots · calibration |
| **Missing values** | Native NaN (XGB/HGB) · median imputation in Pipeline (RF) |
| **Class imbalance** | Balanced sample weights on every estimator |
| **Feature engineering** | 12 physics-motivated features (8 original + 4 new) |
| **Model** | Soft-voting ensemble: XGBoost + HistGradientBoosting + RandomForest |
| **Performance** | ~81.4% accuracy · ~79.0% Macro F1 · ~78.7% CV F1 (±0.7%) |
| **Explainability** | SHAP global bar · beeswarm · per-class waterfall · plain-English narrative |
| **Deployment** | Live Flask web app · REST prediction API · Three.js 3D visualisation |

### What Could Be Improved

- **Optuna hyperparameter tuning** — current parameters are manually set; Bayesian optimisation could add ~1%
- **LightGBM as a 4th ensemble member** — adds model diversity with minimal overhead
- **Platt scaling / isotonic regression** — would improve confidence score reliability
- **Full Kepler DR25 TCE dataset** — includes pipeline-level centroid-shift and difference-image features

### Acknowledgements

- **Dataset:** NASA Exoplanet Archive, KOI Cumulative Table (public domain)
- **SHAP:** Lundberg & Lee, 2017 — *A Unified Approach to Interpreting Model Predictions*
- **XGBoost:** Chen & Guestrin, 2016 — *XGBoost: A Scalable Tree Boosting System*
- **Challenge:** India High School Exoplanet Data Challenge / Celesta

---
*Celesta — Srinath V Venkateshan · vvsrinath0@gmail.com*